## Lecture 4: Acquiring Data and Text as Data

### 1. Karthik's Inspection Data

We read the data exported from the NYC portal after having uploadeded it to `05_lecture/data`.

In [ ]:
import pandas as pd

# "parse_dates" tells pandas which columns have dates (so that we can compute months and years and weeks, say.)
karthik = pd.read_csv("data/NYC_Cooling_Tower_System_Inspection_Results_20261119.csv",parse_dates=["Inspection_date"])
karthik.head()

,BIN,System_ID,Address,Borough,ZIP_Code,Status,Active_Equip,Inspection_date,Violation_Code,Law_Section,...,Citation_Text,Summons_Number,Inspection_Type,BBL,Latitude,Longitude,Community_Board,Council_District,Census_Tract,NTA_Code
0,2013591,2000010644,2432 GRAND CONCOURSE,Bronx,10458.0,Active,1,2024-12-16,NaN,NaN,...,NaN,NaN,Cycle,2031520060,40.860628,-73.898113,205.0,15.0,23702.0,BX0503
1,2117335,2000013846,1250 WATERS PLACE,Bronx,10461.0,Active,1,2024-12-12,NaN,NaN,...,NaN,NaN,Cycle,2042267501,40.852771,-73.838851,211.0,33.0,28400.0,BX1161
2,2117335,2000001902,1250 WATERS PLACE,Bronx,10461.0,Active,1,2024-12-12,NaN,NaN,...,NaN,NaN,Cycle,2042267501,40.852771,-73.838851,211.0,33.0,28400.0,BX1161
3,2094950,2000002024,1407 EAST 180 STREET,Bronx,10460.0,Active,1,2024-12-12,NaN,NaN,...,NaN,NaN,Cycle,2040170001,40.840161,-73.870595,211.0,15.0,24000.0,BX1101
4,2097549,2000010029,1301 MORRIS PARK AVE,Bronx,10461.0,Active,1,2024-12-12,NaN,NaN,...,NaN,NaN,Cycle,2042050003,40.851190,-73.845252,211.0,13.0,29600.0,BX1102


Given the data, let's ask our AI to help us understand inspections in time and place. This is the prompt I used.

>*Insert cells after the highlighted cell I just clicked in the notebook. 
<br>
Use the dataframe "karthik" to create a line plot of the number of critical citations per week over the time covered by the data. 
<br>
Please stick to things I have learned so far.*

Here, the cells in yellow indicate code written by the AI. There is usually a brief explanation of the steps to carry out the task. At the beginning we might just make sure they make sense. Over time, you should spell out what the AI should do.

<div style="background-color:#fff3cd; padding:12px; border-radius:4px">

We want the number of critical citations in each week, so first we mark which rows are critical: `karthik["Violation_Type"] == "Critical"` gives True or False for each row, stored in a new column `is_critical`. Then we count by week. This uses one new tool, `resample("W")`, which groups rows with dates into weeks, much as `groupby()` groups rows by a value. Adding up the True values counts the critical rows in each week, because True counts as 1 and False as 0. We count this way, rather than only looking at the critical rows, so that weeks with no critical citations show up as 0 instead of disappearing. The cell prints some checks: the weekly counts should add up to the number of critical rows.

</div>


In [8]:
# True for a critical violation, False for anything else (including blank)
karthik["is_critical"] = karthik["Violation_Type"] == "Critical"

# make the inspection date the row label, then add up the True values (True counts as 1)
# in each week. resample("W") makes one group per week, ending on a Sunday,
# and weeks with no critical citations are kept, with a count of 0
weekly = karthik.set_index("Inspection_date")["is_critical"].resample("W").sum()

# turn the result into a data frame with two named columns, ready for Altair
weekly = weekly.reset_index()
weekly.columns = ["week_ending", "critical_citations"]

# checks: the weekly counts should add up to the number of critical rows
print("critical rows in the data:", karthik["is_critical"].sum())
print("added up over the weeks:  ", weekly["critical_citations"].sum())
print("number of weeks:", len(weekly))
print("weeks with no critical citations:", (weekly["critical_citations"] == 0).sum())
print("first inspection:", karthik["Inspection_date"].min().date(), " last inspection:", karthik["Inspection_date"].max().date())
print("rows with no inspection date:", karthik["Inspection_date"].isna().sum())
weekly.head()


critical rows in the data: 235
added up over the weeks:   235
number of weeks: 106
weeks with no critical citations: 72
first inspection: 2024-10-03  last inspection: 2026-10-07
rows with no inspection date: 0


,week_ending,critical_citations
0,2024-10-06,0
1,2024-10-13,0
2,2024-10-20,0
3,2024-10-27,0
4,2024-11-03,7


<div style="background-color:#fff3cd; padding:12px; border-radius:4px">

Now we draw the weekly counts as a line chart with Altair, the way we drew bar charts before, but with `mark_line()` instead of `mark_bar()`. The `:T` after `week_ending` tells Altair that this column is a date, so the bottom axis is laid out as a timeline. The `:Q` after `critical_citations` says it is a number.

</div>


In [9]:
import altair as alt

# a line chart: weeks along the bottom, number of critical citations up the side
alt.Chart(weekly).mark_line().encode(
    alt.X("week_ending:T", title="week ending"),
    alt.Y("critical_citations:Q", title="critical citations in the week"),
).properties(title="Critical citations per week", width=700, height=300)


alt.Chart(...)

<div style="background-color:#fff3cd; padding:12px; border-radius:4px">

We want the number of critical citations in each week. In this data a citation is a row with something in the `Violation_Type` column, and "Critical" is one of its values (the others are "General" and "PHH"). This cell keeps only the "Critical" rows. Then `resample("W")` sorts them into weeks by `Inspection_date` and `count()` counts the rows in each week. Each week is labelled by its Sunday. Weeks with no critical citations get a count of 0. The cell prints the number of citations, the number of weeks, how many of those weeks have zero, and the first and last week.

</div>


In [5]:
# keep only the rows that are critical citations
critical = karthik[karthik["Violation_Type"] == "Critical"]
print("critical citations:", len(critical))

# count them by week: resample("W") groups dates into weeks that end on a Sunday
weekly_critical = critical.set_index("Inspection_date").resample("W")["Violation_Type"].count()

# weeks with no critical citations show up as 0, which is what we want for the plot
weekly_critical = weekly_critical.rename("critical_citations").rename_axis("week")

print("weeks:", len(weekly_critical), "| weeks with zero:", (weekly_critical == 0).sum())
print("first week:", weekly_critical.index.min().date(), "| last week:", weekly_critical.index.max().date())
weekly_critical.head()


critical citations: 235
weeks: 102 | weeks with zero: 68
first week: 2024-11-03 | last week: 2026-10-11


week
2024-11-03     7
2024-11-10     0
2024-11-17     3
2024-11-24    14
2024-12-01     5
Freq: W-SUN, Name: critical_citations, dtype: int64

Now, to create a map, we might use a mapping package called folium.

>*Make a street map in folium and overlay 2025 in purple and 2026 in orange at lat/long where there was a critical citation given. Use the stamen basemap.*

Again, yellow roadmap and then some code. 

<div style="background-color:#fff3cd; padding:12px; border-radius:4px">

The map below uses the `folium` package, which may not be installed in your codespace yet. This cell installs it with `%pip install`. It only needs to run once, and it does nothing if `folium` is already there.

</div>

In [26]:
%pip install folium

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


<div style="background-color:#fff3cd; padding:12px; border-radius:4px">

Now a map of where the critical citations were given. `folium` draws a street map that you can zoom and drag. The map pictures (called tiles) come from Stamen's "Toner Lite" style, a plain grey street map that lets the coloured circles stand out. Stamen's tiles are now served by Stadia Maps, so the cell gives `folium.Map` the tile address (`tiles=`) and the credit line to show (`attr=`). We keep the critical rows, work out the year of each inspection with `.dt.year` and split them into 2025 and 2026. Then we loop over the rows and put one circle on the map at each row's `Latitude` and `Longitude`: purple for 2025, orange for 2026. The 2025 circles are drawn first so the 2026 ones sit on top. The circles are see-through, so a spot where several citations were given looks darker. The small control at the top right of the map switches each year on and off, and hovering over a circle shows the address and inspection date. The cell first prints how many citations there are in each year and how many different locations they cover. Citations from 2024 are not on the map, as you asked for 2025 and 2026 only.

</div>


In [12]:
import folium

# the critical citations, with the year of each inspection
crit = karthik[karthik["Violation_Type"] == "Critical"].copy()
crit["year"] = crit["Inspection_date"].dt.year

# 2025 and 2026 each get their own table
crit_2025 = crit[crit["year"] == 2025]
crit_2026 = crit[crit["year"] == 2026]

# checks: how many citations, and how many different places, in each year
print("critical citations in the data:", len(crit), "(2024:", (crit["year"] == 2024).sum(), "- not mapped)")
print("2025: citations", len(crit_2025), "| different locations", crit_2025[["Latitude", "Longitude"]].drop_duplicates().shape[0])
print("2026: citations", len(crit_2026), "| different locations", crit_2026[["Latitude", "Longitude"]].drop_duplicates().shape[0])
print("rows missing a latitude or longitude:", crit[["Latitude", "Longitude"]].isna().any(axis=1).sum())

# the basemap: Stamen's "Toner Lite" street map (Stamen's tiles are now served by Stadia Maps)
stamen_url = "https://tiles.stadiamaps.com/tiles/stamen_toner_lite/{z}/{x}/{y}.png"
stamen_credit = ("&copy; Stadia Maps &copy; Stamen Design &copy; OpenMapTiles "
                 "&copy; OpenStreetMap contributors")

# a street map centred on the middle of all the critical citations
center = [crit["Latitude"].mean(), crit["Longitude"].mean()]
street_map = folium.Map(location=center, zoom_start=12, tiles=stamen_url, attr=stamen_credit)

# one layer per year, so the control at the top right can switch each on and off
layer_2025 = folium.FeatureGroup(name="2025 (purple)")
layer_2026 = folium.FeatureGroup(name="2026 (orange)")

# a purple circle for each 2025 citation
for i, row in crit_2025.iterrows():
    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=6,
        color="purple",
        fill=True,
        fill_color="purple",
        fill_opacity=0.4,
        tooltip=str(row["Address"]) + ", " + str(row["Inspection_date"].date()),
    ).add_to(layer_2025)

# an orange circle for each 2026 citation (drawn after 2025, so on top)
for i, row in crit_2026.iterrows():
    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=6,
        color="orange",
        fill=True,
        fill_color="orange",
        fill_opacity=0.4,
        tooltip=str(row["Address"]) + ", " + str(row["Inspection_date"].date()),
    ).add_to(layer_2026)

layer_2025.add_to(street_map)
layer_2026.add_to(street_map)
folium.LayerControl().add_to(street_map)

street_map


critical citations in the data: 235 (2024: 37 - not mapped)
2025: citations 92 | different locations 34
2026: citations 106 | different locations 42
rows missing a latitude or longitude: 0


The obvious question now is how do we tell if the answer is right or not? 

### 2. Patrick's data

Karthik was lucky because his data were easily accessible. A simple CSV could be exported from a city web site. In Patrick's case, the data are structured into a series of web pages that have to be traversed to reclaim the data. 

Let's describe how this process, so-called web scraping, works.

In [ ]:
# install a couple packages I forgot about when I set up your codespace
%pip install lxml
%pip install bs4

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Here we read a table of data from WikiLeaks using `read_html()`. This function returns not a single dataframe but a list of dataframes, one for each `<table>` on the web page. Patrick's incident reports have just one table per incident.

In [ ]:
# read the grey box of data into a dataframe
patrick = pd.read_html("https://wardiaries.wikileaks.org/id/AA98A562-7D51-413A-91E6-A3F6B5879A19/")

# read_html returns a list with dataframes from all the tables on a web page (10 html tables, 10 list items)
print(type(patrick))
# and in our case there's only one
print(len(patrick))

# here it is
patrick[0]

<class 'list'>
1


,0,1
0,Type,Explosive Hazard
1,Category,IED Explosion
2,Tracking number,2007-033-005438-0943
3,Region,RC EAST
4,Reporting unit,CJSOTF-A
5,Unit name,CJSOTF-A
6,Type of unit,ANSF
7,Attack on,ENEMY
8,MGRS,42SYD1792292079
9,Originator group,UNKNOWN


Now, take the first page of search results and pull the table data into a dataframe, one row per incident.

>*From the list-tag class searchresults on the page 
<br><br>
https://wardiaries.wikileaks.org/search/?sort=date&type_of_unit=ANSF&region=RC+EAST&date__gte=2005-01-01&date__lte=2021-01-01&p=1
<br><br>
create the href url, write a loop over them, pull the table on each incident page (like you just did) and make each a row in a table.*

<div style="background-color:#fff3cd; padding:12px; border-radius:4px">

The grey box we just read belongs to one incident. The site's search page lists many incidents, and we want a particular search: region `RC EAST`, type of unit `ANSF`, dates from 2005 to 2021, sorted by date. All of that is in the address. After the `?` come the settings (`type_of_unit=ANSF`, `region=RC+EAST`, `date__gte=2005-01-01`, `date__lte=2021-01-01`...), and the last one, `p=`, is the page number, so we keep the address without a number (`search_url`) and add the number on the end. Each incident on the page is a list item (`<li>`) in the HTML with `class="searchresult"` (singular, not `searchresults`), and inside it is a link whose `href` is the address of that incident's page, for example `/id/4D0E1E60-.../`. This cell downloads page 1 with `requests`, uses `lxml` and an XPath search (`//li[@class='searchresult']/a/@href`, meaning "the `href` of the link inside every `li` with that class") to pull out the links, and adds the site's address in front of each to make a full URL. The page also says how many results the whole search has, and the cell reads that number out of the text. It prints that number, how many incidents are on this page, and the first three urls. A page shows only 10 incidents.

</div>


In [31]:
import re
import requests
from lxml import html

# the search: region RC EAST, type of unit ANSF, 2005 to 2021, sorted by date
# p= is the page number, so we leave it empty here and add the number when we use it
search_url = ("https://wardiaries.wikileaks.org/search/?sort=date&type_of_unit=ANSF&region=RC+EAST"
              "&date__gte=2005-01-01&date__lte=2021-01-01&p=")

# download page 1 (the site wants us to say what kind of browser we are)
search_page = requests.get(search_url + "1", headers={"User-Agent": "Mozilla/5.0"})
tree = html.fromstring(search_page.text)

# the page says how many results the whole search has, like "2368 results."
total_results = re.search(r"([\d,]+) results", tree.text_content()).group(1)

# the href of the link inside every <li class="searchresult">
hrefs = tree.xpath("//li[@class='searchresult']/a/@href")

# the hrefs start with /id/..., so put the site's address in front to make full urls
incident_urls = ["https://wardiaries.wikileaks.org" + href for href in hrefs]

print("results in the whole search:", total_results)
print("incidents on this page:", len(incident_urls))
print("different urls:", len(set(incident_urls)))
for url in incident_urls[:3]:
    print(url)


results in the whole search: 2368
incidents on this page: 10
different urls: 10
https://wardiaries.wikileaks.org/id/AA98A562-7D51-413A-91E6-A3F6B5879A19/
https://wardiaries.wikileaks.org/id/FD2AF635-A3B2-4898-A48A-3D899AEDC4A2/
https://wardiaries.wikileaks.org/id/1024951/


Now we have them loop over the URLs of search result pages. These correspond to arguments p=2, 3, 4, and so on. 

<div style="background-color:#fff3cd; padding:12px; border-radius:4px">

Now a loop over the urls. For each incident page it does what we did before: `pd.read_html()` reads the grey box, a table with the labels (Type, Category, Region...) in column 0 and the values in column 1. We turn that table into one row by pairing each label with its value (`dict(zip(...))`), add the page's url as an extra entry so we can trace the row back, and add the row to a list. A one-second pause between pages is polite to the website. After the loop, `pd.DataFrame(rows)` makes the table, with one row per incident and one column per label. Any label that one page has and another does not gets a blank (`NaN`). The cell prints some checks: how many rows there are compared with how many urls, whether any page had something other than exactly one table, whether any page repeated a label (a repeat would overwrite the first value), and how many blanks each column has.

</div>


In [32]:
import time

rows = []                 # one dictionary per incident
pages_without_one_table = []
pages_with_repeated_labels = []

for url in incident_urls:
    tables = pd.read_html(url)               # a list of all the tables on the page
    if len(tables) != 1:
        pages_without_one_table.append(url)

    grey_box = tables[0]                     # labels in column 0, values in column 1
    if not grey_box[0].is_unique:
        pages_with_repeated_labels.append(url)

    row = dict(zip(grey_box[0], grey_box[1]))   # {"Type": "Explosive Hazard", "Category": ...}
    row["url"] = url
    rows.append(row)

    time.sleep(1)                            # be polite to the website

incidents = pd.DataFrame(rows)

# checks
print("urls:", len(incident_urls), "| rows in the table:", len(incidents))
print("pages without exactly one table:", len(pages_without_one_table))
print("pages with a repeated label:", len(pages_with_repeated_labels))
print("blank values in each column:")
print(incidents.isna().sum())

incidents


urls: 10 | rows in the table: 10
pages without exactly one table: 0
pages with a repeated label: 0
blank values in each column:
Type                   0
Category               0
Tracking number        0
Region                 0
Reporting unit         0
Unit name              0
Type of unit           0
Attack on              0
MGRS                   1
Originator group       0
Updated by group       0
Affiliation            0
Dcolor                 0
Classification         7
url                    0
Total casualties       8
Host nation killed     8
Civilian killed        9
Host nation wounded    9
dtype: int64


,Type,Category,Tracking number,Region,Reporting unit,Unit name,Type of unit,Attack on,MGRS,Originator group,Updated by group,Affiliation,Dcolor,Classification,url,Total casualties,Host nation killed,Civilian killed,Host nation wounded
0,Explosive Hazard,IED Explosion,2007-033-005438-0943,RC EAST,CJSOTF-A,CJSOTF-A,ANSF,ENEMY,42SYD1792292079,UNKNOWN,JTF Paladin 3rd EOD BN S-3,ENEMY,RED,SECRET,https://wardiaries.wikileaks.org/id/AA98A562-7...,NaN,NaN,NaN,NaN
1,Explosive Hazard,IED Found/Cleared,2007-033-011230-0553,RC EAST,TF SWORD,ANP,ANSF,ENEMY,42SVB66335383,UNKNOWN,J3 ORSA,ENEMY,RED,SECRET,https://wardiaries.wikileaks.org/id/FD2AF635-A...,NaN,NaN,NaN,NaN
2,Explosive Hazard,IED Explosion,1024951,RC EAST,DRUID - ISAF,ANP,ANSF,ENEMY,42SWC9414308687,DRUID - ISAF,J3 ORSA,ENEMY,RED,NaN,https://wardiaries.wikileaks.org/id/1024951/,NaN,NaN,NaN,NaN
3,Explosive Hazard,IED Explosion,1024979,RC EAST,DRUID - ISAF,ANP,ANSF,ENEMY,42SWB836869,DRUID - ISAF,J3 ORSA,ENEMY,RED,NaN,https://wardiaries.wikileaks.org/id/1024979/,NaN,NaN,NaN,NaN
4,Explosive Hazard,Mine Found/Cleared,1024981,RC EAST,DRUID - ISAF,NDS / TF Devil,ANSF,ENEMY,42SXC7594090610,DRUID - ISAF,J3 ORSA,ENEMY,RED,NaN,https://wardiaries.wikileaks.org/id/1024981/,NaN,NaN,NaN,NaN
5,Explosive Hazard,IED Found/Cleared,2007-033-005741-0043,RC EAST,TF DEVIL,ANP / TF DEVIL,ANSF,ENEMY,42SXC181854,UNKNOWN,J3 ORSA,ENEMY,RED,SECRET,https://wardiaries.wikileaks.org/id/1A34DC40-4...,NaN,NaN,NaN,NaN
6,Explosive Hazard,IED Explosion,1025023,RC EAST,DRUID - ISAF,ANA,ANSF,ENEMY,42SWB855875,DRUID - ISAF,J3 ORSA,ENEMY,RED,NaN,https://wardiaries.wikileaks.org/id/1025023/,NaN,NaN,NaN,NaN
7,Explosive Hazard,IED Explosion,1025066,RC EAST,DRUID - ISAF,ANP / TF Devil,ANSF,ENEMY,42SVD8758506369,DRUID - ISAF,J3 ORSA,ENEMY,RED,NaN,https://wardiaries.wikileaks.org/id/1025066/,2,1,1,NaN
8,Explosive Hazard,IED Explosion,1025082,RC EAST,DRUID - ISAF,TF Devil,ANSF,ENEMY,42SXD86136891,DRUID - ISAF,J3 ORSA,ENEMY,RED,NaN,https://wardiaries.wikileaks.org/id/1025082/,19,7,NaN,12
9,Explosive Hazard,IED Found/Cleared,100304,RC EAST,DRUID - No Matches,ANP,ANSF,ENEMY,NaN,DRUID - No Matches,J3 ORSA,ENEMY,RED,NaN,https://wardiaries.wikileaks.org/id/1008651/,NaN,NaN,NaN,NaN


<div style="background-color:#fff3cd; padding:12px; border-radius:4px">

The search page shows 10 incidents at a time, and the whole search has far more than we want to download, so we take pages 2 to 11 as well. This cell loops over the page numbers 2 to 11 (`range(2, 12)` stops before 12). For each number it builds the address (`search_url` plus the number), downloads the page, pulls out the incident links with the same XPath search as before, and adds each one to a list called `more_urls`. The site sometimes lists the same incident twice, so a link we already have is skipped and counted instead. A one-second pause keeps us polite. The cell prints how many links each page had, then how many urls we kept, how many repeats were skipped, and how many were already on page 1 (`incident_urls`), which should be none. To take more pages, change the numbers in `range()`.

</div>


In [33]:
more_urls = []
repeats_skipped = 0

for page_number in range(2, 12):          # pages 2, 3, ... 11
    page_address = search_url + str(page_number)
    page_text = requests.get(page_address, headers={"User-Agent": "Mozilla/5.0"}).text
    page_tree = html.fromstring(page_text)

    page_hrefs = page_tree.xpath("//li[@class='searchresult']/a/@href")
    for href in page_hrefs:
        full_url = "https://wardiaries.wikileaks.org" + href
        if full_url in more_urls or full_url in incident_urls:
            repeats_skipped = repeats_skipped + 1      # the site showed this incident twice
        else:
            more_urls.append(full_url)

    print("page", page_number, ":", len(page_hrefs), "incidents")
    time.sleep(1)                         # be polite to the website

print()
print("new urls:", len(more_urls))
print("different urls:", len(set(more_urls)))
print("repeats skipped:", repeats_skipped)
print("already on page 1:", len(set(more_urls) & set(incident_urls)))


page 2 : 10 incidents
page 3 : 10 incidents
page 4 : 10 incidents
page 5 : 10 incidents
page 6 : 10 incidents
page 7 : 10 incidents
page 8 : 10 incidents
page 9 : 10 incidents
page 10 : 10 incidents
page 11 : 10 incidents

new urls: 100
different urls: 100
repeats skipped: 0
already on page 1: 0


<div style="background-color:#fff3cd; padding:12px; border-radius:4px">

Now the same loop as before, run over the new urls: for each incident page, `pd.read_html()` reads the grey box and we turn it into one row (label: value) with the page's url added. We keep these rows in a list called `new_rows`, build a data frame from them (`new_incidents`), and stack it under the first 10 with `pd.concat()`. `ignore_index=True` renumbers the rows 0, 1, 2... The result is `all_incidents`. This downloads about 100 pages with a one-second pause each, so it takes a couple of minutes. The cell prints checks: rows against urls, pages without exactly one table, pages with a repeated label, whether any url appears twice, and how many blanks each column has. It also checks that the search filter worked: it counts the values in the `Region` and `Type of unit` columns, which should be only `RC EAST` and `ANSF`. Columns that only some incidents have (like the casualty counts) will have many blanks.

</div>


In [34]:
new_rows = []                 # one dictionary per new incident
new_pages_without_one_table = []
new_pages_with_repeated_labels = []

for url in more_urls:
    tables = pd.read_html(url)               # a list of all the tables on the page
    if len(tables) != 1:
        new_pages_without_one_table.append(url)

    grey_box = tables[0]                     # labels in column 0, values in column 1
    if not grey_box[0].is_unique:
        new_pages_with_repeated_labels.append(url)

    row = dict(zip(grey_box[0], grey_box[1]))
    row["url"] = url
    new_rows.append(row)

    time.sleep(1)                            # be polite to the website

new_incidents = pd.DataFrame(new_rows)

# stack the new rows under the first page and renumber the rows
all_incidents = pd.concat([incidents, new_incidents], ignore_index=True)

# checks
print("new urls:", len(more_urls), "| new rows:", len(new_incidents))
print("all rows:", len(all_incidents), "(should be", len(incident_urls) + len(more_urls), ")")
print("pages without exactly one table:", len(new_pages_without_one_table))
print("pages with a repeated label:", len(new_pages_with_repeated_labels))
print("urls that appear twice:", all_incidents["url"].duplicated().sum())
print()
print("did the search filter work? (these should be only RC EAST and ANSF)")
print(all_incidents["Region"].value_counts(dropna=False))
print(all_incidents["Type of unit"].value_counts(dropna=False))
print()
print("blank values in each column:")
print(all_incidents.isna().sum())

all_incidents.head(15)


new urls: 100 | new rows: 100
all rows: 110 (should be 110 )
pages without exactly one table: 0
pages with a repeated label: 0
urls that appear twice: 0

did the search filter work? (these should be only RC EAST and ANSF)
Region
RC EAST    110
Name: count, dtype: int64
Type of unit
ANSF    110
Name: count, dtype: int64

blank values in each column:
Type                     0
Category                 0
Tracking number          0
Region                   0
Reporting unit           0
Unit name                3
Type of unit             0
Attack on                0
MGRS                     1
Originator group         0
Updated by group         0
Affiliation              0
Dcolor                   0
Classification          11
url                      0
Total casualties        90
Host nation killed     100
Civilian killed        107
Host nation wounded     95
CCIR                   108
Sigact                 108
Enemy detained         104
Enemy killed           105
Civilian wounded       107
F

,Type,Category,Tracking number,Region,Reporting unit,Unit name,Type of unit,Attack on,MGRS,Originator group,...,Total casualties,Host nation killed,Civilian killed,Host nation wounded,CCIR,Sigact,Enemy detained,Enemy killed,Civilian wounded,Friendly wounded
0,Explosive Hazard,IED Explosion,2007-033-005438-0943,RC EAST,CJSOTF-A,CJSOTF-A,ANSF,ENEMY,42SYD1792292079,UNKNOWN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Explosive Hazard,IED Found/Cleared,2007-033-011230-0553,RC EAST,TF SWORD,ANP,ANSF,ENEMY,42SVB66335383,UNKNOWN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Explosive Hazard,IED Explosion,1024951,RC EAST,DRUID - ISAF,ANP,ANSF,ENEMY,42SWC9414308687,DRUID - ISAF,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Explosive Hazard,IED Explosion,1024979,RC EAST,DRUID - ISAF,ANP,ANSF,ENEMY,42SWB836869,DRUID - ISAF,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Explosive Hazard,Mine Found/Cleared,1024981,RC EAST,DRUID - ISAF,NDS / TF Devil,ANSF,ENEMY,42SXC7594090610,DRUID - ISAF,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Explosive Hazard,IED Found/Cleared,2007-033-005741-0043,RC EAST,TF DEVIL,ANP / TF DEVIL,ANSF,ENEMY,42SXC181854,UNKNOWN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Explosive Hazard,IED Explosion,1025023,RC EAST,DRUID - ISAF,ANA,ANSF,ENEMY,42SWB855875,DRUID - ISAF,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Explosive Hazard,IED Explosion,1025066,RC EAST,DRUID - ISAF,ANP / TF Devil,ANSF,ENEMY,42SVD8758506369,DRUID - ISAF,...,2,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Explosive Hazard,IED Explosion,1025082,RC EAST,DRUID - ISAF,TF Devil,ANSF,ENEMY,42SXD86136891,DRUID - ISAF,...,19,7,NaN,12,NaN,NaN,NaN,NaN,NaN,NaN
9,Explosive Hazard,IED Found/Cleared,100304,RC EAST,DRUID - No Matches,ANP,ANSF,ENEMY,NaN,DRUID - No Matches,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


MGRS specify locations (the Military Grid Reference System). The package `mgrs` turns them into latitude and longitude. Here is my mapping prompt.

>*Use the mgrs to create lat.long coordinates, add lat and long as two columns to the `all_incidents` data frame. then use the stamen basemap again to create a map of purple colored dots, one for each incident.*

This can be refined, we can add a slider to vary time, there are lots of things to do next.

In [27]:
%pip install mgrs

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


<div style="background-color:#fff3cd; padding:12px; border-radius:4px">

The `MGRS` column gives each incident's location in the Military Grid Reference System, for example `42SYD1792292079`: a grid zone (`42S`), a 100 km square (`YD`), and then digits that say how far east and north of the square's corner the incident is. A map needs latitude and longitude instead, so we convert with the `mgrs` package: `mgrs.MGRS().toLatLon(code)` returns the pair (latitude, longitude). Some codes have spaces in them (`38S MB 078074`), so we take the spaces out first. The function `mgrs_to_lat_long()` returns blanks (`NaN`) for any code the package cannot read, rather than stopping. We loop over the `MGRS` column and store the results in two new columns of `all_incidents`, `lat` and `long`.

The cell then checks the result. It prints how many rows got no coordinates, lists those rows, and prints the smallest and largest latitude and longitude. Every incident in this search is in region `RC EAST`, so the coordinates should all be in one part of the world, and a code that converted to somewhere far from the others would be a warning sign.

</div>


In [35]:
import mgrs

converter = mgrs.MGRS()

def mgrs_to_lat_long(code):
    """'38S MB 078074' -> (latitude, longitude); (nan, nan) if the code cannot be read"""
    try:
        lat, long = converter.toLatLon(str(code).replace(" ", ""))
        return lat, long
    except Exception:
        return float("nan"), float("nan")

# convert every code, then put the two numbers in their own columns
pairs = [mgrs_to_lat_long(code) for code in all_incidents["MGRS"]]
all_incidents["lat"] = [pair[0] for pair in pairs]
all_incidents["long"] = [pair[1] for pair in pairs]

# checks
print("incidents:", len(all_incidents))
print("rows with no coordinates:", all_incidents["lat"].isna().sum())
print("latitude from", all_incidents["lat"].min().round(2), "to", all_incidents["lat"].max().round(2))
print("longitude from", all_incidents["long"].min().round(2), "to", all_incidents["long"].max().round(2))
print()

# the rows that could not be converted
print(all_incidents[all_incidents["lat"].isna()][["Region", "Category", "MGRS", "url"]].to_string())

all_incidents[["Region", "Category", "MGRS", "lat", "long"]].head(10)


incidents: 110
rows with no coordinates: 2
latitude from 32.03 to 35.15
longitude from 67.94 to 71.39

     Region           Category            MGRS                                                                        url
9   RC EAST  IED Found/Cleared             NaN                               https://wardiaries.wikileaks.org/id/1008651/
59  RC EAST  IED Found/Cleared  42SWB371004320  https://wardiaries.wikileaks.org/id/8473ED9A-C67A-4DDB-B2C4-506E942703FE/


/tmp/ipykernel_10004/228497480.py:8: RuntimeWarning: Warning in "Convert_MGRS_To_Geodetic": Latitude Warning
  lat, long = converter.toLatLon(str(code).replace(" ", ""))


,Region,Category,MGRS,lat,long
0,RC EAST,IED Explosion,42SYD1792292079,35.148033,71.392229
1,RC EAST,IED Found/Cleared,42SVB66335383,33.022464,68.639532
2,RC EAST,IED Explosion,42SWC9414308687,33.513606,70.013683
3,RC EAST,IED Explosion,42SWB836869,33.318435,69.898686
4,RC EAST,Mine Found/Cleared,42SXC7594090610,34.241735,70.910492
5,RC EAST,IED Found/Cleared,42SXC181854,34.203379,70.282427
6,RC EAST,IED Explosion,42SWB855875,33.323697,69.919153
7,RC EAST,IED Explosion,42SVD8758506369,34.398671,68.864934
8,RC EAST,IED Explosion,42SXD86136891,34.945670,71.038303
9,RC EAST,IED Found/Cleared,NaN,NaN,NaN


<div style="background-color:#fff3cd; padding:12px; border-radius:4px">

Now the map. It uses the same Stamen "Toner Lite" tiles as the citations map (`stamen_url` and `stamen_credit` come from that cell, so run it first). First we keep only the incidents that have coordinates (`dropna(subset=...)` drops any rows we could not convert). Then we loop over those rows and put one purple, see-through circle at each row's `lat` and `long`, with a tooltip showing the incident's category and tracking number when you hover. Instead of choosing a centre and zoom ourselves we use `fit_bounds()`, which zooms the map to just contain the smallest and largest latitude and longitude. The cell prints how many dots it drew.

</div>


In [36]:
import folium

# only the incidents that have coordinates
located = all_incidents.dropna(subset=["lat", "long"])
print("incidents:", len(all_incidents), "| dots on the map:", len(located))

# the Stamen street map (stamen_url and stamen_credit were set in the citations map cell)
incident_map = folium.Map(tiles=stamen_url, attr=stamen_credit)

# one purple circle for each incident
for i, row in located.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["long"]],
        radius=5,
        color="purple",
        fill=True,
        fill_color="purple",
        fill_opacity=0.5,
        tooltip=str(row["Category"]) + ", " + str(row["Tracking number"]),
    ).add_to(incident_map)

# zoom the map to just contain all the dots
incident_map.fit_bounds([
    [located["lat"].min(), located["long"].min()],
    [located["lat"].max(), located["long"].max()],
])

incident_map


incidents: 110 | dots on the map: 108


### Exercise

We are going to try to recreate [the arrow shift plot from NBC](https://www.nbcnews.com/politics/2026-special-elections/ohio-senate-results). What steps do we have to follow to make this?

The course notes cover how we found the data and stored it in `ohio-senate-results.json`. This prompt then forms a table of shifts and then creates the plot we want.

>*Use the `ohio-senate-results.json` in data under 05_lecture to make a vote shift matrix. I'd like to make the image on https://www.nbcnews.com/politics/2026-special-elections/ohio-senate-results with the left-right arrows. I will need county, shift and party the shift went to.*

In [7]:
from requests import get

response = get("https://www.nbcnews.com/firecracker/api/v2/state-results/2026-special-elections/ohio-senate-results")
shift = response.json()

response.status_code

200

<div style="background-color:#fff3cd; padding:12px; border-radius:4px">

NBC's vote shift graphic compares each county's result in this Senate race with the same county in the 2024 presidential election. The **shift** is the change in the margin between the two parties, in percentage points, and each county gets an arrow that points toward the party the county moved toward. To draw it we need one row per county with three things: the `county`, the `shift` (how many points) and the `party` the shift went to.

The file `data/ohio-senate-results.json` is a saved copy of the data behind NBC's page, so we read it from disk instead of downloading it again. `json.load()` turns the file into nested dictionaries and lists. The county shifts are under `races`, then the first race (`[0]`), then `voteShiftCounties`, a list with one dictionary per county. Inside each one, `marginPastRaces[0]` holds the comparison with the past race: `marginShift` is the number of points and `marginShiftParty` is the party it moved toward (`dem` or `rep`). We loop over the counties and collect one dictionary per county, then make a data frame with `pd.DataFrame()`.

We also keep each county's margin now and in 2024, because that lets us check NBC's numbers. A margin is how far ahead the leading party is, so we turn the margins into one scale where Republican leads are positive and Democratic leads are negative. The shift is then the margin now minus the margin in 2024, and its sign tells us which way the county moved. The cell recomputes every shift and party that way and counts how many differ from the ones in the file. It also prints the number of counties, how many shifted toward each party, the smallest and largest shifts, the statewide shift stored in the file, and how much of the vote had been counted in each county when the file was saved (`percent_in`). The table at the end has just the three columns we need, with the biggest shifts first.

</div>


In [3]:
import json
import pandas as pd

# read the saved copy of NBC's data
with open("data/ohio-senate-results.json") as file:
    ohio = json.load(file)

# one dictionary per county, in a list
counties = ohio["races"][0]["voteShiftCounties"]

rows = []
for county in counties:
    past = county["marginPastRaces"][0]      # the comparison with the 2024 presidential election
    rows.append({
        "county": county["name"],
        "shift": past["marginShift"],        # points the margin moved
        "party": past["marginShiftParty"],   # party the shift went to: dem or rep
        "margin_now": county["margin"],
        "party_now": county["marginLeadingParty"],
        "margin_2024": past["margin"],
        "party_2024": past["leadingParty"],
        "percent_in": county["percentIn"],   # percent of the vote counted when the file was saved
    })
vote_shift = pd.DataFrame(rows)

# check: put Republican leads as positive and Democratic leads as negative,
# then the shift is (margin now) minus (margin in 2024). Positive means toward Republicans.
sign = {"rep": 1, "dem": -1}
now = vote_shift["margin_now"] * vote_shift["party_now"].map(sign)
then = vote_shift["margin_2024"] * vote_shift["party_2024"].map(sign)
change = now - then

my_shift = change.abs().round(2)
my_party = change.apply(lambda c: "rep" if c > 0 else "dem")

# allow 0.011 for rounding to two decimal places
wrong_shift = (my_shift - vote_shift["shift"]).abs() > 0.011
wrong_party = my_party != vote_shift["party"]

print("counties:", len(vote_shift), "| different county names:", vote_shift["county"].nunique())
print("shifts that differ from my calculation:", wrong_shift.sum())
print("parties that differ from my calculation:", wrong_party.sum())
print("counties that shifted toward each party:", vote_shift["party"].value_counts().to_dict())
print("smallest shift:", vote_shift["shift"].min(), "| largest shift:", vote_shift["shift"].max())
print("statewide shift in the file:", ohio["races"][0]["voteShiftSummary"])
print("percent of the vote counted, county by county: from", vote_shift["percent_in"].min(), "to", vote_shift["percent_in"].max())

# the table we need: county, shift and the party the shift went to, biggest shifts first
shift_table = vote_shift[["county", "shift", "party"]].sort_values("shift", ascending=False).reset_index(drop=True)
shift_table


counties: 88 | different county names: 88
shifts that differ from my calculation: 0
parties that differ from my calculation: 0
counties that shifted toward each party: {'dem': 88}
smallest shift: 9.47 | largest shift: 22.48
statewide shift in the file: {'marginShift': 16.06, 'marginShiftParty': 'dem'}
percent of the vote counted, county by county: from 92.6 to 99.0


,county,shift,party
0,Scioto,22.48,dem
1,Lawrence,22.46,dem
2,Pike,22.42,dem
3,Vinton,22.14,dem
4,Jackson,22.13,dem
...,...,...,...
83,Auglaize,11.17,dem
84,Delaware,11.06,dem
85,Hamilton,10.47,dem
86,Mercer,9.74,dem


<div style="background-color:#fff3cd; padding:12px; border-radius:4px">

To draw the shifts on a map we need the shape of each Ohio county. The file `data/cb_2018_us_county_20m.zip` holds the outlines of every county in the US, and `gpd.read_file()` can read it straight from the zip. Each county has a `STATEFP` (the state's code; Ohio's is `39`) and a `GEOID` (a code for the county), so we keep the rows where `STATEFP` is `"39"`.

Matching the counties by name is risky (spellings can differ), so we match by code instead. NBC's file gives a `fips_code` for each county (`voteShiftMapUnits`), and it is the same code as `GEOID` in the shapes. We add that code to our `vote_shift` table, then `merge()` joins the two tables: each shape picks up the `shift` and `party` of the county with the same code. We also work out the middle of each county, where its arrow will go. "Middle" is not reliable in latitude and longitude, so we work it out on a flat map measured in metres (`epsg=32617`, a projection that suits Ohio) and then convert the point back to latitude and longitude (`epsg=4326`), saved as `lat` and `long`.

The cell then checks the join: how many shapes there are (Ohio has 88 counties), whether any shape missed out on a shift, whether any NBC code has no shape, whether the county names in the two files agree, and whether each middle point really falls inside its county.

</div>


In [4]:
import geopandas as gpd

# every county in the US; geopandas can read the zip file directly
us_counties = gpd.read_file("data/cb_2018_us_county_20m.zip")

# keep Ohio: its state code is 39
ohio_map = us_counties[us_counties["STATEFP"] == "39"].copy()

# NBC's file has a code for each county (fips_code). Make a dictionary: county name -> code
fips_by_county = {}
for unit in ohio["races"][0]["voteShiftMapUnits"]:
    fips_by_county[unit["county_name"]] = unit["fips_code"]

# add the code to our shift table
vote_shift["fips"] = vote_shift["county"].map(fips_by_county)

# join: each shape picks up the shift and party of the county with the same code
ohio_map = ohio_map.merge(vote_shift[["fips", "county", "shift", "party"]],
                          left_on="GEOID", right_on="fips", how="left")

# the middle of each county, worked out on a flat map in metres, then turned back into lat and long
flat = ohio_map.to_crs(epsg=32617)
middles = flat.centroid
middles_lat_long = middles.to_crs(epsg=4326)
ohio_map["long"] = middles_lat_long.x
ohio_map["lat"] = middles_lat_long.y

# checks
print("shapes:", len(ohio_map), "(Ohio has 88 counties)")
print("shapes with no shift:", ohio_map["shift"].isna().sum())
print("NBC codes with no shape:", len(set(vote_shift["fips"]) - set(ohio_map["GEOID"])))
print("county names that differ between the two files:", (ohio_map["NAME"] != ohio_map["county"]).sum())
print("middle points inside their own county:", flat.contains(middles).sum(), "of", len(ohio_map))
print("shapes are in:", us_counties.crs.name)

ohio_map[["GEOID", "NAME", "county", "shift", "party", "lat", "long"]].head()


shapes: 88 (Ohio has 88 counties)
shapes with no shift: 0
NBC codes with no shape: 0
county names that differ between the two files: 0
middle points inside their own county: 88 of 88
shapes are in: NAD83


,GEOID,NAME,county,shift,party,lat,long
0,39153,Summit,Summit,15.60,dem,41.124480,-81.531255
1,39137,Putnam,Putnam,9.47,dem,41.020976,-84.133692
2,39051,Fulton,Fulton,14.23,dem,41.602252,-84.125732
3,39113,Montgomery,Montgomery,14.48,dem,39.753929,-84.290492
4,39173,Wood,Wood,14.74,dem,41.362215,-83.622974


<div style="background-color:#fff3cd; padding:12px; border-radius:4px">

Now the map, with Altair. It has two layers stacked with `+`. The first layer, `mark_geoshape()`, draws the 88 county outlines in light grey. The second layer, `mark_text()`, puts an arrow character at each county's `long` and `lat`. NBC's arrows point diagonally: `↖` (up and to the left, blue) for a shift to Democrats and `↗` (up and to the right, red) for a shift to Republicans. The `arrow` column holds that character, chosen from `party` with `map()`. The arrow's size comes from the county's `shift`, so a bigger arrow is a bigger shift, and its colour comes from `party`: blue for Democrats, red for Republicans. Hovering over an arrow shows the county, the shift and the party. `.project(type="mercator")` tells Altair how to flatten the map, and both layers share it, so the arrows land on their counties.

</div>

In [6]:
import altair as alt

# an arrow for each county: up-left for a shift to Democrats, up-right for a shift to Republicans
ohio_map["arrow"] = ohio_map["party"].map({"dem": "↖", "rep": "↗"})

# layer 1: the county outlines
outlines = alt.Chart(ohio_map[["county", "geometry"]]).mark_geoshape(
    fill="#f2f2f2", stroke="#999999", strokeWidth=0.5
)

# layer 2: one arrow per county, at its middle. The arrow is bigger for a bigger shift.
arrow_data = ohio_map[["county", "shift", "party", "arrow", "long", "lat"]]
arrows = alt.Chart(arrow_data).mark_text(fontWeight="bold").encode(
    longitude="long:Q",
    latitude="lat:Q",
    text="arrow:N",
    size=alt.Size("shift:Q", scale=alt.Scale(domain=[0, 25], range=[8, 32]), title="shift (points)"),
    color=alt.Color("party:N",
                    scale=alt.Scale(domain=["dem", "rep"], range=["#0471E6", "#DE3535"]),
                    title="shifted toward"),
    tooltip=["county", "shift", "party"],
)

# stack the two layers and flatten the map
(outlines + arrows).project(type="mercator").properties(
    title="Ohio Senate special election: shift in margin from the 2024 presidential vote",
    width=550,
    height=550,
)

alt.LayerChart(...)

### 4. Text as Data

Today we have been looking at text-based data and how we acquire that data. Our sub-goal is to use AI to help us analyze data. So one common exercise is to translate free text into a category. For the Guaardian data, we might categorize whether or not the daters wanted a second date. The answer to that questioin is free text and it can be a little tricky to work out. So... we had the AI read each answer and categorize the response as Yes; Yes, as a friend; No; or no indication either way. 

We can, of course question these categories too. But let's start there. The AI labled all 200 of the dates. How do we test what it's done?

In [8]:
import pandas as pd

# this sets a display option for data frames - usually you see just 10 columns and this way you see them all
pd.set_option("display.max_columns", None)
# this specifies up to 100 characters displayable in each cell
pd.set_option("display.max_colwidth",200)

dates = pd.read_csv("data/small_dates.csv")
dates.head()

,name1,age1,score1,name2,age2,score2,firstPublicationDate,standfirst,trailText,bodyText,body
0,Leon,45,9.0,Lucy,40,8.7,2026-11-14T06:00:38Z,"<p>Lucy, 40, a general manager, meets Leon, 45, an operations manager at a food charity</p>","Lucy, 40, a general manager, meets Leon, 45, an operations manager at a food charity","Leon on Lucy What were you hoping for? I’m a regular reader and have always wanted to be in the column, so mostly here for the vibes. Bumping into the love of my life would be nice, though … First...","<h2>Leon on Lucy</h2>\n<p><strong>What were you hoping for?</strong>\n <br>\n I’m a regular reader and have always wanted to be in the column, so mostly here for the vibes. Bumping into the love o..."
1,William,28,9.0,Abigail,24,7.0,2026-11-07T06:00:02Z,"<p>William, 28, a PhD candidate, meets Abigail, 24, a theatre artist</p>","William, 28, a PhD candidate, meets Abigail, 24, a theatre artist",William on Abigail What were you hoping for? To have a spark of some kind. First impressions? She looked very glamorous and suited the smart restaurant far more than I did. What did you talk about...,<h2><strong>William on Abigail</strong></h2>\n<p><strong>What were you hoping for?</strong>\n <br>\n To have a spark of some kind.</p>\n<p><strong>First impressions?</strong>\n <br>\n She looked v...
2,Nelson,41,9.0,Louise,37,8.0,2026-10-31T06:00:38Z,"<p>Nelson, 41, who works in legal services, meets Louise, 37, who works in marketing</p>","Nelson, 41, who works in legal services, meets Louise, 37, who works in marketing","Nelson on Louise What were you hoping for? Great company and wonderful conversation. First impressions? Louise is such a sweet, sincere person. With very interesting hobbies! What did you talk abo...",<h2>Nelson on Louise</h2>\n<p><strong>What were you hoping for?</strong>\n <br>\n <strong> </strong>Great company and wonderful conversation.</p>\n<p><strong>First impressions?</strong>\n <br>\n L...
3,Alexandrea,32,8.0,Pardis,32,8.0,2026-10-24T05:00:26Z,"<p>Alexandrea, 32, a lawyer, meets Pardis, 32, a doctor</p>","Alexandrea, 32, a lawyer, meets Pardis, 32, a doctor",Alexandrea on Pardis What were you hoping for? A fun date and to hopefully meet the one! First impressions? Handsome and relaxed. What did you talk about? The sports we do. Books. Family. Friends....,<p><em> </em></p>\n<h2>Alexandrea on Pardis</h2>\n<p><strong>What were you hoping for?</strong>\n <br>\n A fun date and to hopefully meet the one!</p>\n<p><strong>First impressions?</strong>\n <br...
4,James,34,7.5,Michelle,30,7.0,2026-10-17T05:00:23Z,"<p>James, 34, a researcher, meets Michelle, 30, a radio presenter </p>","James, 34, a researcher, meets Michelle, 30, a radio presenter","James on Michelle What were you hoping for? An unforgettable evening. First impressions? Glamorous. What did you talk about? Birmingham, as we both went to university there, but at different times...",<h2>James on Michelle</h2>\n<p><strong>What were you hoping for?</strong>\n <br>\n An unforgettable evening.</p>\n<p><strong>First impressions?</strong>\n <br>\n Glamorous.</p>\n<p><strong>What di...


<div style="background-color:#fff3cd; padding:12px; border-radius:4px">

**Adding `again_1` and `again_2`.** Each date ends with the question "Would you meet again?", answered once by each person. `again_1` is the answer of the first person (`name1`) and `again_2` the answer of the second (`name2`). I read all 386 answers myself and gave each one a category: **"yes"**, **"no"**, **"as friends"**, or **NA** (blank) when the answer does not actually say. The rules I used:

- **"as friends"** if the answer says they would meet but not romantically ("as friends", "as mates", "platonically"), including "maybe as friends" and "yes, but as friends".
- **"yes"** if they say they would, or describe a plan (a second date, a concert, a walk, a dinner).
- **"no"** if they say no, "probably not", "unlikely", or decline, and I also counted a few where the no is clear but unstated, like "I'm going to continue my search".
- **NA** if the answer says nothing about meeting again ("we swapped numbers"), is unsure ("maybe", "possibly", "we'll see"), depends on chance ("we might bump into each other"), or is a joke.

These are judgement calls and someone else might draw the lines differently. The dictionary below has a note on many of the borderline ones. It is keyed by the two names, since that is how the dates are identified here, and the last lines copy the labels into `dates`.

</div>

In [9]:
# I read all 386 answers (193 dates x 2 daters) by hand. Each line is
#   (name1, name2): (again_1, again_2)
# again_1 is what name1 said, again_2 is what name2 said.
# None means NA. A comment after a line is a note on a borderline call.
hand_again = {
    ('Leon', 'Lucy'): (None, 'yes'),  # swapped numbers, hang out in same places / open to it
    ('William', 'Abigail'): (None, None),  # swapped numbers only / leaves it to him
    ('Nelson', 'Louise'): ('yes', 'as friends'),
    ('Alexandrea', 'Pardis'): ('as friends', 'as friends'),
    ('James', 'Michelle'): ('yes', 'yes'),
    ('Graeme', 'Holly'): ('yes', None),  # we'll see
    ('Shahar', 'Stephen'): (None, None),  # chatted about maybe going / let's see
    ('Miklos', 'Minnie'): (None, 'no'),  # missing a spark, not about meeting
    ('Dec', 'Dave'): (None, 'as friends'),  # swapped numbers, no spark / friendly chemistry, stay in touch
    ('Joe', 'Jade'): (None, 'yes'),  # swapped numbers only
    ('Don', 'Claire'): ('yes', None),  # hopefully / no spark but more conversation to be had
    ('Mike', 'Caterina'): (None, 'as friends'),  # may meet for coffee, open invitation
    ('Susan', 'Pete'): (None, None),  # no spark, not about meeting / might go for a walk, conditional
    ('Andrew', 'Gilli'): ('no', 'as friends'),  # another date pointless / may meet up, no relationship potential
    ('Imogen', 'Andy'): ('yes', 'yes'),  # implied
    ('Richard', 'Katrina'): ('yes', 'as friends'),  # possibly, as friends
    ('Jacob', 'Sophie'): ('as friends', 'as friends'),
    ('Hugh', 'Edie'): ('yes', 'as friends'),
    ('Suli', 'Abigo'): ('yes', 'yes'),
    ('Lily', 'Brodie'): ('no', 'no'),
    ('Philip', 'Carol'): ('yes', None),  # swapped numbers only
    ('Alex', 'Ellie'): ('as friends', 'no'),
    ('Yusuf', 'Hannah'): ('as friends', 'as friends'),  # absolutely, although probably as friends / potentially, as friends
    ('Laurine', 'Theo'): ('as friends', 'as friends'),
    ('Mike', 'Ailsa'): (None, None),  # about connection only / about connection only
    ('Colman', 'Ben'): ('yes', 'yes'),
    ('Frances', 'Eddie'): (None, None),  # distance might prove prohibitive / different stage, swapped numbers
    ('Jonathan', 'Katie'): ('as friends', None),  # maybe in five years, conditional
    ('Josh', 'Hannah'): ('yes', 'yes'),  # plans to see a film / I hope so
    ('Abby', 'Charlie'): ('as friends', 'no'),  # maybe as friends / implied: different things in a partner
    ('Sophie', 'Rebecca'): ('as friends', None),  # maybe at her comedy shows
    ('Jack', 'Heather'): ('as friends', 'no'),  # yes, maybe as friends / implied: age gap insurmountable
    ('Rachel', 'Josh'): ('as friends', 'yes'),
    ('Adam', 'Tina'): ('yes', 'yes'),
    ('Brian', 'Jeanette'): ('yes', 'no'),  # implied: best person to go karaoke with
    ('Robin', 'Jacquie'): ('no', 'no'),
    ('Harry', 'Freya'): ('yes', 'yes'),  # I hope we do
    ('Jack', 'Brigitte'): ('as friends', 'as friends'),  # might just be as friends
    ('Freya', 'Greg'): ('yes', 'yes'),
    ('Sabah', 'David'): ('yes', None),  # motivated to give readers a story
    ('Alfie', 'Sam'): ('as friends', 'yes'),
    ('Ursula', 'Culann'): ('yes', 'yes'),  # dates suggested
    ('Toby', 'Liam'): (None, 'as friends'),  # lack of spark, not about meeting
    ('Jenny', 'Sara'): (None, 'yes'),  # happy to see her if she visits, conditional / implied: love to see her show
    ('Dara', 'Alexia'): ('as friends', 'yes'),
    ('Karyshma', 'Arun'): ('yes', 'yes'),  # happy to see how things develop
    ('Dan', 'Emmie'): ('yes', 'yes'),  # implied: invited him along
    ('Aaron', 'Tara'): (None, 'yes'),  # would say hi if paths crossed / implied: love to go to his gigs
    ('Rita', 'Tom'): ('no', 'no'),
    ('Paul', 'Amanda'): ('yes', 'yes'),
    ('Sarah', 'Russell'): ('as friends', 'yes'),  # maybe as friends / implied: exchanged numbers to visit
    ('Henry', 'Sarah'): (None, 'no'),  # will bump into each other, not a plan / probably not
    ('Alex', 'Rachel'): ('as friends', 'as friends'),
    ('Will', 'Fred'): (None, 'yes'),  # mentioned a second date
    ('Alexi', 'Oisín'): (None, None),  # no spark, not about meeting / friend vibes, not about meeting
    ('Daniel', 'Isla'): ('yes', 'yes'),
    ('Emma', 'Julien'): ('no', 'no'),  # implied: stars not aligned / unlikely
    ('Hannah', 'Ioan'): ('yes', 'as friends'),
    ('Jenny', 'Nova'): ('yes', None),  # I think we may
    ('Kammy', 'Eliza'): ('as friends', 'yes'),  # implied: come to pub quiz
    ('Niamh', 'Harry'): (None, 'as friends'),  # he has my number
    ('Martha', 'Jack'): ('no', 'no'),  # probably not / implied: different kind of person
    ('Joe', 'Mabyn'): ('yes', None),  # friends or something more remains to be seen
    ('Jane', 'Bryn'): ('yes', 'yes'),
    ('Taylor', 'Joshua'): ('no', 'yes'),
    ('Jonah', 'Sunny'): ('as friends', None),  # for sure, but possibly just as friends / will ask about racing, not about meeting
    ('Billy', 'Phoebe'): ('as friends', 'no'),  # no date, but up for a catch-up
    ('Alex', 'Zoe'): ('yes', 'as friends'),  # yes, but maybe more as friends
    ('Imelda', 'Jayesh'): ('yes', 'as friends'),  # promised to revisit restaurant / going back, as friends
    ('Edward', 'Matthew'): (None, None),  # will bump into each other / will bump into him
    ('Rebecca', 'Michael'): ('no', 'no'),  # she declined
    ('Andrew', 'Susan'): ('yes', None),  # talked about meeting again / maybe
    ('Tope', 'Eden'): ('yes', 'yes'),
    ('Annice', 'Benjy'): (None, None),  # swapped numbers, messaged / maybe a second date
    ('Matthew', 'Emma'): ('yes', 'no'),
    ('Liaqat', 'Karim'): ('as friends', 'as friends'),  # probably as friends
    ('Silva', 'Megan'): ('yes', 'as friends'),  # had a nice vibe as friends
    ('Scarlett', 'Issy'): (None, 'as friends'),  # platonic vibes, might go climbing, we shall see
    ('William', 'Lucy'): ('yes', 'no'),
    ('Imogen', 'Sam'): (None, 'as friends'),  # different places
    ('Denise', 'Federico'): ('no', 'as friends'),  # I doubt it / work-related instead
    ('Rachael', 'George'): ('yes', 'yes'),  # Friday on the cards
    ('Kat', 'Matt'): ('yes', 'yes'),
    ('Bridget', 'Sacha'): ('as friends', 'yes'),  # friendship vibes only / hopes to tempt her with a tour
    ('Monique', 'Ian'): ('yes', 'yes'),
    ('Henry', 'Anna'): ('yes', 'yes'),  # on the cards, unsure if more than friends
    ('Emily', 'Jack'): (None, None),  # hard not to meet, live nearby / we will see
    ('Adrian', 'Catherine'): ('yes', 'yes'),
    ('Lewis', 'Max'): ('yes', 'yes'),
    ('Yukari', 'John'): ('yes', 'yes'),  # glad to share another dessert, little in common
    ('Gráinne', 'Alexander'): ('yes', 'yes'),  # made a bet about dinner / I hope so
    ('Matthew', 'Izzy'): ('yes', 'yes'),  # second date discussed / spoke about dinner again
    ('Ben', 'Maxim'): ('yes', 'yes'),
    ('Holly', 'Johnny'): ('yes', 'yes'),
    ('Sadiye', 'Iwan'): ('no', 'no'),
    ('Clemmie', 'Joe'): ('yes', None),  # bump into each other / likely to bump into each other
    ('Jordan', 'Frances'): ('yes', 'as friends'),  # perhaps as friends
    ('Felix', 'Erin'): ('yes', None),  # not about meeting
    ('Ash', 'Darren'): ('yes', 'yes'),
    ('Joe', 'Alison'): ('yes', 'yes'),
    ('Abhijit', 'Alanna'): ('as friends', 'as friends'),
    ('Tom', 'Francoise'): ('as friends', 'as friends'),
    ('Katy', 'Nick'): ('as friends', 'yes'),
    ('David', 'Eva'): ('yes', 'yes'),  # decent possibility, party invitation / invited him to party
    ('Sarah', 'Max'): (None, 'yes'),  # joke answer
    ('Florence', 'Joe'): ('as friends', None),  # maybe as friends / maybe not, up for it in principle
    ('Alan', 'Robert'): ('yes', 'yes'),
    ('Sophie', 'Jack'): ('as friends', 'as friends'),
    ('Richard', 'Rosa'): ('as friends', 'as friends'),  # up for it, maybe as friends
    ('Rashpal', 'Chloe'): ('yes', 'yes'),  # probably / happily
    ('Lauren', 'Ben'): ('no', 'yes'),  # happy to leave it there
    ('Angela', 'Phil'): ('yes', 'yes'),  # welcome in Liverpool any time
    ('Sophie', 'Harry'): (None, 'as friends'),  # not sure paths would cross / yes, but probably not romantic
    ('Julia', 'Isabella'): ('as friends', 'as friends'),
    ('Emi', 'Hari'): ('no', 'yes'),  # hang out again for sure
    ('Elliot', 'Jordan'): ('as friends', 'yes'),  # as mates
    ('Ashley', 'Eva'): (None, 'as friends'),  # she'd think about it
    ('Sonia', 'Richard'): ('no', 'as friends'),  # going to continue search
    ('Ella', 'John'): ('no', None),  # pretty unlikely / if in same neighbourhood, who knows
    ('Anwit', 'Kassondra'): ('yes', 'yes'),
    ('Birgit', 'Solomon'): ('yes', None),  # I'll let you know
    ('Nina', 'Grace'): (None, None),  # quite possibly / we'll see
    ('Suzie', 'Toby'): ('yes', 'no'),  # up for a drink / probably not
    ('Nima', 'Maxim'): ('as friends', 'yes'),  # perhaps more as friends / hopefully a coffee
    ('Ollie', 'Simone'): ('yes', 'yes'),
    ('Peter', 'Linda'): (None, 'yes'),  # in touch, who knows / we said we would
    ('Rebecca', 'Lars'): ('yes', 'as friends'),  # for a pint / definitely as friends, if not something more
    ('Agnes', 'Tom'): ('yes', 'as friends'),
    ('Myrna', 'Manos'): ('no', 'no'),  # chemistry wasn't there / not likely
    ('Laura', 'Paddy'): ('yes', 'as friends'),  # romantically no, would catch up
    ('Anona', 'Nadeem'): ('as friends', None),  # yes as friends / swapped numbers only
    ('Maz', 'Gianluca'): ('as friends', 'as friends'),  # maybe as friends
    ('Scott', 'Maria'): ('yes', 'yes'),
    ('Trisha', 'Neil'): (None, 'yes'),  # talked about a concert, let's see
    ('Harry', 'Jack'): ('as friends', 'yes'),  # maybe more as friends
    ('Sonia', 'Stefan'): (None, 'yes'),  # very possibly
    ('Sam', 'Zillah'): ('yes', 'as friends'),  # no romantic sparks / platonically
    ('Hywel', 'Leah'): (None, 'as friends'),  # paths may cross at yoga / maybe as a friend
    ('Jordy', 'Luke'): ('yes', 'yes'),
    ('Kirstie', 'Tom'): ('yes', 'yes'),
    ('Todd', 'Alicia'): ('no', None),  # probably not / who knows, might bump into each other
    ('Ariel', 'Alex'): ('no', 'no'),
    ('Susan', 'Gerard'): ('no', 'no'),
    ('Raina', 'Tony'): ('yes', 'yes'),
    ('Sajeda', 'Fred'): ('yes', 'yes'),  # probably
    ('Margaret', 'Dee'): ('yes', None),  # any time / could be slow-burner, different worlds
    ('Andy', 'Donna'): ('yes', 'as friends'),  # drink or meal
    ('Angelica', 'Peter'): (None, None),  # swapped numbers only / possibly
    ('Deborah', 'Mike'): ('no', 'yes'),  # probably not
    ('Mark', 'Abby'): ('yes', 'no'),  # not averse / yes as friends was a joke, no for dating
    ('Emily', 'Shivani'): ('yes', 'yes'),  # promised a trip
    ('Edwin', 'Marjo'): ('yes', 'as friends'),
    ('George', 'Stephen'): ('yes', None),  # let's see
    ('Andrew', 'Alex'): (None, 'yes'),  # a boy can dream / for sure
    ('Jayson', 'Eleanor'): ('no', 'no'),
    ('Harry', 'Eileen'): ('yes', 'yes'),
    ('Lara', 'Claire'): ('yes', 'no'),
    ('Adam', 'Rachel'): ('yes', 'yes'),  # planning session next week
    ('Lucy', 'Adrian'): ('no', 'no'),  # wished each other farewell
    ('Peter', 'Nick'): (None, 'yes'),  # you never know / second date or as friends
    ('Robert', 'David'): ('no', 'yes'),  # very unlikely
    ('Lennard', 'Sophia'): (None, 'yes'),  # possibly
    ('Ruth', 'Stacy'): ('yes', 'yes'),  # hopefully
    ('Zander', 'Rachel'): (None, 'as friends'),  # we'll see
    ('Tim', 'Katherine'): ('yes', None),  # not about meeting
    ('Niko', 'Alya'): ('as friends', 'no'),
    ('Harry', 'River'): (None, 'yes'),  # yeah maybe
    ('Marc', 'Tanya'): ('yes', 'as friends'),
    ('Jake', 'Alessia'): ('as friends', 'no'),  # yes, but not romantic
    ('Kiana', 'Gabriel'): ('no', 'as friends'),  # probably not / perhaps as friends
    ('Ruben', 'Jem'): ('yes', 'as friends'),  # love to, maybe more as friends
    ('Beatriz', 'Julie-Yara'): ('as friends', 'as friends'),
    ('Harry', 'Jessie'): ('yes', None),  # swapped numbers only
    ('Shelagh', 'Tony'): ('yes', 'yes'),  # why not
    ('Karin', 'Lloyd'): (None, 'yes'),  # possibly
    ('Stephanie', 'Simon'): ('as friends', 'yes'),  # wouldn't be against the idea
    ('Philipp', 'Thomas'): ('as friends', 'as friends'),  # not on a date, drinks with friends / perhaps as friends
    ('Sophie', 'Alex'): ('yes', None),  # we'll see
    ('Emily', 'Ned'): ('yes', None),  # maybe
    ('JD', 'Oli'): ('yes', 'yes'),  # will text back
    ('Tim', 'Patrick'): ('yes', 'yes'),
    ('Max', 'Kajol'): ('yes', 'yes'),
    ('Charley', 'Matt'): ('as friends', 'no'),  # not romantically, fun mate
    ('Seán', 'Sam'): ('yes', 'yes'),
    ('David', 'Jacqueline'): ('no', 'no'),
    ('Alison', 'Emma'): (None, None),  # not sure paths will cross / might bump into her
    ('Katie', 'Will'): ('as friends', 'no'),  # of course, probably as friends
    ('Zula', 'Leyla'): (None, None),  # might bump into each other / would say hi
    ('Giles', 'Sarah'): ('yes', 'yes'),
    ('Dora', 'Jago'): ('yes', 'yes'),
    ('Will', 'Reece'): ('yes', 'yes'),  # I hope so
    ('Alec', 'Shitong'): ('yes', 'as friends'),  # if she wants to / probably, as friends
    ('Deborah', 'Graham'): ('yes', 'yes'),
}

# look up each date by its two names to fill the new columns
dates["again_1"] = [hand_again[(a, b)][0] for a, b in zip(dates["name1"], dates["name2"])]
dates["again_2"] = [hand_again[(a, b)][1] for a, b in zip(dates["name1"], dates["name2"])]

# how many of each answer? (None shows up as NaN)
print(dates["again_1"].value_counts(dropna=False))
print(dates["again_2"].value_counts(dropna=False))
dates[["name1", "name2", "again_1", "again_2"]].head(10)


again_1
yes           91
NaN           39
as friends    39
no            24
Name: count, dtype: int64
again_2
yes           82
as friends    46
NaN           36
no            29
Name: count, dtype: int64


,name1,name2,again_1,again_2
0,Leon,Lucy,NaN,yes
1,William,Abigail,NaN,NaN
2,Nelson,Louise,yes,as friends
3,Alexandrea,Pardis,as friends,as friends
4,James,Michelle,yes,yes
5,Graeme,Holly,yes,NaN
6,Shahar,Stephen,NaN,NaN
7,Miklos,Minnie,NaN,no
8,Dec,Dave,NaN,as friends
9,Joe,Jade,NaN,yes


In [10]:
dates.columns

Index(['name1', 'age1', 'score1', 'name2', 'age2', 'score2',
       'firstPublicationDate', 'standfirst', 'trailText', 'bodyText', 'body',
       'again_1', 'again_2'],
      dtype='str')

<div style="background-color:#fff3cd; padding:12px; border-radius:4px">

**Check.** My own reading could be wrong, so this cell compares it with a second, much cruder method. For each date it pulls the two answers to "Would you meet again?" out of the `body` html, then guesses a category from keywords: any mention of "friend", "mate" or "platonic" counts as "as friends"; then words like "no", "probably not" or "unlikely" count as "no"; then words like "yes", "definitely" or "I would" count as "yes"; otherwise NA. The keyword guess is bad at sarcasm, jokes and "yes, but..." answers, so we should not expect the two methods to agree everywhere. The cell prints the totals, a table of my labels against the keyword guesses, and every answer where both methods picked a category but picked different ones, so you can read those and decide who is right.

</div>

In [11]:
import re

# pull the two answers to "Would you meet again?" out of the html in the body column
def get_answers(body):
    pieces = re.split(r"Would you meet again\??", body)[1:]   # the text after each question
    answers = []
    for piece in pieces:
        piece = piece.split("</p>")[0]            # stop at the end of the paragraph
        piece = re.sub(r"<[^>]+>", "", piece)     # remove any html tags
        piece = piece.replace("&nbsp;", " ").strip()
        answers.append(piece)
    return answers

# a crude keyword guess at the category, checked in this order
def keyword_guess(answer):
    text = answer.lower()
    if re.search(r"friend|mate|platonic", text):
        return "as friends"
    if re.search(r"\bno\b|not likely|probably not|don.t think so|unlikely|i doubt", text):
        return "no"
    if re.search(r"\byes\b|sure|definitely|absolutely|of course|hopefully|planning|we plan|i would|i hope", text):
        return "yes"
    return None

# one row per answer: the text, my hand label and the keyword guess
rows = []
for i in range(len(dates)):
    answers = get_answers(dates.loc[i, "body"])
    hand_labels = [dates.loc[i, "again_1"], dates.loc[i, "again_2"]]
    for j in range(2):
        rows.append({"row": i, "daters": j + 1, "answer": answers[j],
                     "hand": hand_labels[j], "keyword": keyword_guess(answers[j])})
check = pd.DataFrame(rows)

# the allowed categories and the totals
print("answers checked:", len(check), "(should be", 2 * len(dates), ")")
print(pd.concat([dates["again_1"], dates["again_2"]]).value_counts(dropna=False))

# hand labels (down) against keyword guesses (across); None is shown as "NA"
print()
print(pd.crosstab(check["hand"].fillna("NA"), check["keyword"].fillna("NA")))

# the disagreements that matter: both methods picked a category, and they differ
both = check[check["hand"].notna() & check["keyword"].notna() & (check["hand"] != check["keyword"])]
print()
print("answers where both picked a category but differ:", len(both))
for _, r in both.iterrows():
    print(f'row {r["row"]}, daters {r["daters"]}: hand = {r["hand"]}, keywords = {r["keyword"]}\n    {r["answer"]}')


answers checked: 386 (should be 386 )
yes           173
as friends     85
NaN            75
no             53
Name: count, dtype: int64

keyword     NA  as friends  no  yes
hand                               
NA          60           4   2    9
as friends   3          78   2    2
no          13           5  33    2
yes         57           3   1  112

answers where both picked a category but differ: 15
row 35, daters 2: hand = no, keywords = yes
    We live about four hours apart, so we aren’t planning to meet again.
row 42, daters 2: hand = as friends, keywords = yes
    For a run and a coffee, sure, but not romantically.
row 48, daters 1: hand = no, keywords = yes
    He’s a cat lover and I’m allergic. So I think it’s best I bow out now – I would hate to make him have to choose.
row 51, daters 2: hand = no, keywords = as friends
    Probably not, I don’t think there was a romantic spark – it was all very friendly.
row 56, daters 2: hand = no, keywords = as friends
    We didn’t swap 

How do we verify what the AI did? Don't forget this code... it might help. 

In [ ]:
from IPython.display import HTML

# use the function HTML() to show the data in the 5th row of the "body" column
# of our dates DataFrame

HTML(dates["body"][5])

### 5. Regulations again

We now have 150,000 cases (comments) and it might be a touch expensive to have the AI read them all. What do we do? The AI turns to code, learning ad hoc models for first and last names and locations. What do you think?

In [ ]:
from IPython.display import HTML

# use the function HTML() to show the data in the 6th row of the "body" column
# of our dates DataFrame

HTML(dates["body"][5])

In [13]:
import pandas as pd

reg = pd.read_csv("data/sample_regulation.csv")
reg.shape

(15000, 60)

In [14]:
# let tables show up to 70 characters of text in each cell
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.max_columns", None)

In [15]:
reg.columns

Index(['Document ID', 'Agency ID', 'Docket ID', 'Tracking Number',
       'Document Type', 'Posted Date', 'Is Withdrawn?',
       'Federal Register Number', 'FR Citation', 'Title', 'Comment Start Date',
       'Comment Due Date', 'Allow Late Comments', 'Comment on Document ID',
       'Effective Date', 'Implementation Date', 'Postmark Date',
       'Received Date', 'Author Date', 'Related RIN(s)', 'Authors', 'CFR',
       'Abstract', 'Legacy ID', 'Media', 'Document Subtype',
       'Exhibit Location', 'Exhibit Type', 'Additional Field 1',
       'Additional Field 2', 'Topics', 'Duplicate Comments',
       'OMB/PRA Approval Number', 'Page Count', 'Page Length', 'Paper Width',
       'Special Instructions', 'Source Citation', 'Start End Page', 'Subject',
       'First Name', 'Last Name', 'City', 'State/Province', 'Zip/Postal Code',
       'Country', 'Organization Name', 'Submitter Representative',
       'Representative's Address', 'Representative's City, State & Zip',
       'Government

In [16]:
reg

,Document ID,Agency ID,Docket ID,Tracking Number,Document Type,Posted Date,Is Withdrawn?,Federal Register Number,FR Citation,Title,Comment Start Date,Comment Due Date,Allow Late Comments,Comment on Document ID,Effective Date,Implementation Date,Postmark Date,Received Date,Author Date,Related RIN(s),Authors,CFR,Abstract,Legacy ID,Media,Document Subtype,Exhibit Location,Exhibit Type,Additional Field 1,Additional Field 2,Topics,Duplicate Comments,OMB/PRA Approval Number,Page Count,Page Length,Paper Width,Special Instructions,Source Citation,Start End Page,Subject,First Name,Last Name,City,State/Province,Zip/Postal Code,Country,Organization Name,Submitter Representative,Representative's Address,"Representative's City, State & Zip",Government Agency,Government Agency Type,Comment,Category,Restrict Reason Type,Restrict Reason,Reason Withdrawn,Content Files,Attachment Files,"Display Properties (Name, Label, Tooltip)"
0,USBC-2026-0628-0015,USBC,USBC-2026-0628,mtv-qe2l-fv7a,Public Submission,2026-09-15T04:00Z,False,NaN,NaN,Comment on FR Doc # 2026-18481,NaN,NaN,False,USBC-2026-0628-0001,NaN,NaN,NaN,2026-09-10T04:00Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,My family comes from the Pickstown/Fort Randall Dam area of South ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,USBC-2026-0628-0032,USBC,USBC-2026-0628,mtv-zs9w-487n,Public Submission,2026-09-15T04:00Z,False,NaN,NaN,Comment on FR Doc # 2026-18481,NaN,NaN,False,USBC-2026-0628-0001,NaN,NaN,NaN,2026-09-10T04:00Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Re: Docket USBC-2026-0628\nDecennial Census of the Population of A...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,USBC-2026-0628-0034,USBC,USBC-2026-0628,mtw-5qq2-3840,Public Submission,2026-09-15T04:00Z,False,NaN,NaN,Comment on FR Doc # 2026-18481,NaN,NaN,False,USBC-2026-0628-0001,NaN,NaN,NaN,2026-09-10T04:00Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Department of Commerce, Census Bureau\nUSBC-2026-0628\nRIN 0607-AA...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,USBC-2026-0628-0053,USBC,USBC-2026-0628,mtw-h75y-vn0r,Public Submission,2026-09-15T04:00Z,False,NaN,NaN,Comment on FR Doc # 2026-18481,NaN,NaN,False,USBC-2026-0628-0001,NaN,NaN,NaN,2026-09-11T04:00Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"In Texas, even immigrants use the roads. So we need to count immig...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,USBC-2026-0628-0059,USBC,USBC-2026-0628,mtw-xwdr-y86v,Public Submission,2026-09-15T04:00Z,False,NaN,NaN,Comment on FR Doc # 2026-18481,NaN,NaN,False,USBC-2026-0628-0001,NaN,NaN,NaN,2026-09-11T04:00Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ever since the beginning of the census immigration status has not ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14995,USBC-2026-0628-99967,USBC,USBC-2026-0628,mvk-dm8k-gagc,Public Submission,2026-10-29T04:00Z,False,NaN,NaN,Comment on FR Doc # 2026-18481,NaN,NaN,False,USBC-2026-0628-0001,NaN,NaN,NaN,2026-10-22T04:00Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"U.S Census Bureau,The proposed census rule would change who is inc...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
14996,USBC-2026-0628-99969,USBC,USBC-2026-0628,mvk-dnvb-bs8i,Public Submission,2026-10-29T04:00Z,False,NaN,NaN,Comment on FR Doc # 2026-18481,NaN,NaN,False,USBC-2026-0628-0001,NaN,NaN,NaN,2026-10-22T04:00Z,NaN,N

<div style="background-color:#fff3cd; padding:12px; border-radius:4px">

If many people copy the same form letter, their comments will start with the same words. So this cell takes the first 200 characters of each comment, in lower case, and adds them to `reg` as a new column called `opening`. Then `value_counts()` counts how many comments have each opening. The table shows the 10 openings used by the most comments, with the first 60 characters of each so it fits on screen. It also prints how many different openings there are among the 15,000 comments.

</div>

In [17]:
# let tables show up to 70 characters of text in each cell
pd.set_option("display.max_colwidth", 200)

# the first 200 characters of each comment, in lower case
reg["opening"] = reg["Comment"].str.strip().str[:200].str.lower()

# how many comments have each opening? (most common first)
opening_counts = reg["opening"].value_counts()

print("comments:", len(reg))
print("different openings:", len(opening_counts))

# the 10 most common openings, shortened to 60 characters so the table fits
top10 = opening_counts.head(10).reset_index()
top10.columns = ["opening", "comments"]
top10["opening"] = top10["opening"].str[:200].str.replace("\n", " ")
top10

comments: 15000
different openings: 2529


,opening,comments
0,"dear decennial census of the population of americans; proposed residence criteria and proposed regulations for demographic questions, subject: re: decennial census of the population of americans;...",1721
1,"us census bureau,i urge the census bureau and department of commerce to withdraw the proposed rule changing residence criteria for the decennial census and restricting demographic questions.the pr...",1657
2,"the u.s. constitution requires the census to count “the whole number of persons in each state” every ten years—not the whole number of citizens, and not only the people the administration finds co...",1338
3,the u.s. census bureau's proposed rule to strengthen the us census is critical to fair representation and the future of america. the u.s. census should not include distorted data in its count o...,821
4,i am writing to comment on the proposed rule “decennial census of the population of americans; proposed residence criteria and proposed regulations for demographic questions” (docket no. 260903-00...,741
5,to the u.s. census bureau: i’m writing to comment on the proposed rule “decennial census of the population of americans; proposed residence criteria and proposed regulations for demographic quest...,355
6,"add introduction text here. if introduction text is not needed, uncheck the above checkbox ""include introduction"". this text is not editable by the supporter. i oppose the proposed changes to the...",297
7,"dear decennial census of the population of americans; proposed residence criteria and proposed regulations for demographic questions, i strongly oppose the proposed rule's changes to the residenc...",283
8,"u.s census bureau,i strongly oppose the newly proposed census rule. it would drastically change who is included in the count and where people are counted, which in turn will impact political repre...",271
9,"u.s census bureau,i am urging the census bureau to reject the newly proposed census rule. it would not only exclude people based on their immigration status, it would also ban questions about race...",265


<div style="background-color:#fff3cd; padding:12px; border-radius:4px">

The table above lists only the 10 biggest openings. This cell looks at all 2,529 of them. It sorts the openings by how many comments use each one: 1 comment (the opening is unique), 2 to 4, 5 to 49, and 50 or more. For each group the table gives the number of openings, the number of comments they hold, and the share of all 15,000 comments. `pd.cut()` does the sorting into groups, and `groupby()` adds up the comments in each.

</div>

In [18]:
# put each opening into a group by how many comments use it
group_names = ["1 comment", "2 to 4 comments", "5 to 49 comments", "50 or more comments"]
opening_group = pd.cut(opening_counts, [0, 1, 4, 49, 100000], labels=group_names)

# how many openings fall in each group, and how many comments those openings hold
openings_in_group = opening_group.value_counts().sort_index()
comments_in_group = opening_counts.groupby(opening_group, observed=True).sum()

group_table = pd.DataFrame({
    "openings": openings_in_group,
    "comments": comments_in_group,
})
group_table["share of all comments (%)"] = (group_table["comments"] / len(reg) * 100).round(1)
group_table

,openings,comments,share of all comments (%)
count,,,
1 comment,2424,2424,16.2
2 to 4 comments,44,103,0.7
5 to 49 comments,23,486,3.2
50 or more comments,38,11987,79.9


<div style="background-color:#fff3cd; padding:12px; border-radius:4px">

This cell draws a bar chart of the 20 most common openings, with the number of comments using each one as the bar length. Each label starts with the opening's rank (1 is the most common) and then its first 30 characters, so two openings that start alike can still be told apart. The bars are sorted from the biggest at the top. Hover over a bar to read the whole 200-character opening.

</div>

In [19]:
import altair as alt

# the 20 most common openings and how many comments use each
top20 = opening_counts.head(20).reset_index()
top20.columns = ["opening", "comments"]

# a short label for the axis: rank, then the first 30 characters (new lines turned into spaces)
top20["rank"] = range(1, 21)
top20["label"] = top20["rank"].astype(str) + ". " + top20["opening"].str[:30].str.replace("\n", " ")

# horizontal bars, biggest at the top
alt.Chart(top20).mark_bar().encode(
    alt.X("comments:Q", title="number of comments with this opening"),
    alt.Y("label:N", sort=alt.EncodingSortField("rank"), title=None),
    tooltip=["rank", "comments", "opening"],
).properties(title="The 20 most common openings (first 200 characters)", width=500, height=400)

alt.Chart(...)

<div style="background-color:#fff3cd; padding:12px; border-radius:4px">

This cell shows the last 110 characters of 100 randomly chosen comments, because a signature, if there is one, comes at the end. `reg.sample(100, random_state=7)` picks the 100 rows; the `random_state` makes it pick the same 100 every time. The number at the start of each line is the row number in `reg`. I read these 100 myself and counted how many end with a full name (first and last), how many with only a first name, and how many with no name at all. Read them yourself and see whether you agree.

</div>

In [20]:
# pick 100 comments at random (the same 100 every time, because of random_state)
pick = reg.sample(100, random_state=7)

# the last 110 characters of each, with new lines turned into spaces
endings = pick["Comment"].str.strip().str.replace("\n", " ").str[-110:]

# print the row number and the ending of each comment
for row_number, ending in endings.items():
    print(row_number, "...", ending)

404 ... priately, etc.  I strongly oppose this proposed rule and encourage the Census Bureau to withdraw the proposal.
5221 ... áspora sean contadas, una vez y en el lugar correcto.  Gracias por considerar este comentario.  Gerardo Vicéns
3357 ... his proposed rule so the 2030 Census fairly and accurately counts our community.    Sincerely, Thomas Hall PhD
7336 ... December 9th, and to reject this proposed rule.    David Friscia, 6[X] Spanaway WA    Sincerely, David Friscia
8313 ... nd preserve a Census that fairly and accurately counts our community.Rachel Godbout[x]Madison, Wisconsin 53703
7716 ... lation. Such outcomes are unacceptable and would harm all Americans.  Sincerely, Ms. Susan Ludke Pemberton, NJ
5943 ... apportionment counts. I urge you to withdraw this rule and protect the integrity of the census.KarenCalifornia
14689 ... d preserve a Census that fairly and accurately counts our community.Helane  [X] San Clemente, California 92672
100 ... his proposed rule.  Enough is En

<div style="background-color:#fff3cd; padding:12px; border-radius:4px">

Many comments end with a signature, but there is no standard layout. The 100 endings above show the main ones: `Sincerely, Mary Jones Boise, ID` and `...our community.Mary Jones [X] Boise, Idaho 83702` (where `[X]` is a tick-box left by the comment form), and `Thank you, Mary Jones`. The cells below turn these layouts into three new columns: `com_first`, `com_last` and `com_place`. They set up the lists and patterns, define the helper functions, define `get_signature()`, add the columns to `reg`, and then check the result against a hand reading. This first cell only sets things up. It lists the states (spelled out and as two-letter codes) and the titles (Mr, Dr...) that are not part of a name, and builds the search patterns we will use to find them. Nothing is added to `reg` yet.

</div>

In [21]:
import re

# the states, spelled out and as two-letter codes, so we can recognise places like "Boise, ID 83702"
state_names = ["Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado", "Connecticut", "Delaware",
               "Florida", "Georgia", "Hawaii", "Idaho", "Illinois", "Indiana", "Iowa", "Kansas", "Kentucky",
               "Louisiana", "Maine", "Maryland", "Massachusetts", "Michigan", "Minnesota", "Mississippi",
               "Missouri", "Montana", "Nebraska", "Nevada", "New Hampshire", "New Jersey", "New Mexico",
               "New York", "North Carolina", "North Dakota", "Ohio", "Oklahoma", "Oregon", "Pennsylvania",
               "Rhode Island", "South Carolina", "South Dakota", "Tennessee", "Texas", "Utah", "Vermont",
               "Virginia", "Washington", "West Virginia", "Wisconsin", "Wyoming", "District of Columbia",
               "Puerto Rico"]
state_codes = ["AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "FL", "GA", "HI", "ID", "IL", "IN", "IA", "KS",
               "KY", "LA", "ME", "MD", "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH", "NJ", "NM", "NY",
               "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC", "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV",
               "WI", "WY", "DC", "PR"]

# words that sit next to a name but are not part of it (Mr, Dr, Jr, PhD...)
titles = {"mr", "mrs", "ms", "miss", "mx", "dr", "rev", "prof", "hon", "spc", "sgt", "capt", "col"}
suffixes = {"jr", "sr", "ii", "iii", "iv", "phd", "md", "esq"}

# ordinary words: if one of these turns up where we expect a name, we picked up a sentence by mistake
not_names = {"the", "of", "to", "for", "i", "you", "thank", "thanks", "please", "we", "my", "this", "that",
             "is", "are", "it", "a", "an", "in", "on", "at", "by", "with", "as", "be", "not", "no", "yes",
             "census", "bureau", "proposal", "rule", "attached", "see", "comment", "regards", "sincerely"}

# search patterns (regular expressions). The longest state names go first so "West Virginia" beats "Virginia"
all_states = "|".join(re.escape(s) for s in sorted(state_names + state_codes, key=len, reverse=True))
spelled_states = "|".join(re.escape(s) for s in sorted(state_names, key=len, reverse=True))

# "Sincerely," "Respectfully ~" and so on
sign_off = re.compile(r"(?:Sincerely|Respectfully|Respectively)[,.]?\s*~?\s*", re.I)
# "Thank you," "Thanks." "Regards," and so on (used to find where a name starts)
any_closing = re.compile(r"(?:Sincerely|Respectfully|Respectively|Thank you|Thanks|Regards)"
                         r"(?: for [a-z ]+?)?[,.!~]*\s*", re.I)
# text that ends with a state and maybe a zip code, like "Mary Jones Boise, ID 83702"
ends_with_state = re.compile(r"^(.*?)[,\s]*\b(" + all_states + r")\b\.?\s*(\d{5}(?:-\d{4})?)?\s*$", re.I)
# a first name stuck to a state with no space, like "RoxanneOhio"
name_then_state = re.compile(r"(?:(?<=[a-z.!?)])|\s)([A-Z][a-z]{2,})(" + spelled_states + r")\s*$")

print(len(state_names), "state names,", len(state_codes), "state codes")

52 state names, 52 state codes


<div style="background-color:#fff3cd; padding:12px; border-radius:4px">

This cell defines four small helper functions that the main function in the next cell will use. `clean_text()` removes HTML tags, `&nbsp;`, template placeholders such as `[Your information here]` and extra spaces, but keeps the `[X]` tick-box. `drop_titles()` takes a list of words and removes titles (Mr, Dr...), suffixes (Jr, PhD...) and any `[X]`, leaving just the name. `split_name_and_place()` takes text like `Mary Jones Boise, ID 83702` and splits it into the name (`Mary Jones`) and the place (`Boise, ID 83702`): the first two words that are not single letters are the name, and whatever is left before the state is the city. `name_before_marker()` finds a name that comes right before an `[X]` by cutting the text back to the last sentence end or closing such as `Thank you,`. Defining a function does not run it, so the cell only prints a short message to say it worked.

</div>

In [22]:
def clean_text(text):
    """remove html tags, [Your information here] style placeholders, &nbsp; and extra spaces"""
    text = re.sub(r"<[^>]+>", " ", str(text))
    text = re.sub(r"\[[^\]]{2,}\]", " ", text)
    text = text.replace("&nbsp;", " ")
    return re.sub(r"\s+", " ", text).strip()


def is_initial(word):
    """True for a single letter like 'B' or 'B.'"""
    return len(word.strip(".")) == 1


def drop_titles(words):
    """take a list of words and leave out titles, suffixes and the [X] tick-box"""
    keep = []
    for word in words:
        word = word.strip(",;~")
        plain = word.strip(".").lower()
        if re.fullmatch(r"\[[xX]\]\W*", word):
            continue
        if word and plain not in titles and plain not in suffixes:
            keep.append(word)
    return keep


def split_name_and_place(tail):
    """'Mary Jones Boise, ID 83702' -> (['Mary', 'Jones'], 'Boise, ID 83702')"""
    found = ends_with_state.match(tail)
    if not found:
        return drop_titles(tail.split()), None      # no state, so treat it all as the name

    words = drop_titles(found.group(1).replace(",", " ").split())

    # the name is the first two words that are not single letters ("Linda and Milo Rivera" is four words)
    if len(words) > 2 and words[1].lower() in ("and", "&"):
        name_length = min(len(words), 4)
    else:
        name_length, real_words = 0, 0
        while name_length < len(words) and real_words < 2:
            if not is_initial(words[name_length]):
                real_words = real_words + 1
            name_length = name_length + 1

    city = " ".join(words[name_length:])
    state = found.group(2) + (" " + found.group(3) if found.group(3) else "")
    place = (city + ", " + state) if city else state
    return words[:name_length], place


def name_before_marker(before):
    """the name is the last few words before the [X]: cut at the last sentence end or closing"""
    before = before.rstrip(" ,.~")
    cuts = [m.end() for m in re.finditer(r"(?<=[a-z]{3})[.!?]\s*", before)]        # "...proposal. Mary"
    cuts += [m.end() for m in re.finditer(r"[.,!?~:;)](?=[A-Za-z])", before)]      # "...you,Mary"
    cuts += [m.end() for m in any_closing.finditer(before)]                        # "Thank you, Mary"
    segment = before[max(cuts):] if cuts else before
    words = drop_titles(segment.split())

    # some people's names are typed twice: "Liz Rice Janzen Liz Rice Janzen"
    half = len(words) // 2
    if len(words) % 2 == 0 and half >= 2 and [w.lower() for w in words[:half]] == [w.lower() for w in words[half:]]:
        words = words[:half]
    return words

print("helper functions defined")

helper functions defined


<div style="background-color:#fff3cd; padding:12px; border-radius:4px">

This cell defines `get_signature()`, which takes one comment and returns three things: first name, last name and place (or `None` where it finds nothing). It looks for the layouts we saw, in this order: (1) an `[X]` and a "Sincerely," closing after it, as in `Mary Jones, [X] Boise ID  Sincerely, Mary Jones`; (2) an `[X]` close after a closing word, as in `Respectfully ~Mary Jones [X] Boise ID`; (3) an `[X]` on its own, as in `...community.Mary Jones [X] Boise, Idaho 83702`; (4) a "Sincerely," near the end, as in `Sincerely, Mary Jones Boise, ID`; (5) a short "Thank you," near the end, or a first name stuck to a state like `MaryIdaho`. If the words it found look like ordinary sentence text (more than 6 words, a number, or words like "the" or "census"), it returns no name. A last word that is a single letter (`Rebecca T`) is treated as an initial, so there is a first name but no last name. Names keep the capital letters the writer used.

</div>

In [23]:
def get_signature(comment):
    """return (first name, last name, place) written at the end of a comment; None where not found"""
    text = clean_text(comment)
    marker = re.search(r"\[[xX]\]", text)                  # the tick-box, written [X] or [x]
    sign_offs = list(sign_off.finditer(text))
    last_sign_off = sign_offs[-1] if sign_offs else None   # the last "Sincerely," in the text

    name_words, place = [], None

    if marker and last_sign_off and last_sign_off.start() > marker.start():
        # "Mary Jones, [X] Boise ID    Sincerely, Mary Jones"
        name_words = drop_titles(text[last_sign_off.end():].split())
        place = text[marker.end():last_sign_off.start()]

    elif marker and last_sign_off and 0 <= marker.start() - last_sign_off.end() < 60:
        # "Respectfully ~Mary Jones Boise ID [X]"
        name_words, place = split_name_and_place(text[last_sign_off.end():marker.start()])
        if place is None:
            place = text[marker.end():]

    elif marker:
        # "...our community.Mary Jones [X] Boise, Idaho 83702"
        name_words = name_before_marker(text[:marker.start()])
        place = text[marker.end():]

    elif last_sign_off and len(text) - last_sign_off.end() < 80:
        # "Sincerely, Mary Jones Boise, ID"
        name_words, place = split_name_and_place(text[last_sign_off.end():])

    else:
        thank_yous = list(re.finditer(r"Thank(?:s| you)[,.!]?\s*", text, re.I))
        stuck = name_then_state.search(text)
        if thank_yous and 0 < len(text[thank_yous[-1].end():].split()) <= 4 and len(text) - thank_yous[-1].end() < 40:
            # "Thank you, Mary Jones"
            name_words, place = split_name_and_place(text[thank_yous[-1].end():])
        elif stuck:
            # "...the census.MaryIdaho"
            name_words, place = [stuck.group(1)], stuck.group(2)

    # tidy the place: no stray commas, nothing after "USA", no second copy of the zip code
    if place is not None:
        place = re.sub(r"\s+,", ",", place).strip(" ,.")
        place = re.sub(r"\s*,?\s*USA.*$", "", place)
        place = re.sub(r"(\d{5}(?:-\d{4})?)(?=[A-Za-z]).*$", r"\1", place).strip(" ,.")
        if place == "" or len(place) > 60:
            place = None

    # if what we picked up looks like a sentence or does not start with a real name, there is no name
    looks_wrong = len(name_words) > 6
    for word in name_words:
        if word.lower().strip(".,") in not_names or re.search(r"\d", word):
            looks_wrong = True
    if len(name_words) == 0 or looks_wrong or is_initial(name_words[0]):
        return None, None, place

    first = name_words[0]
    last = None
    if len(name_words) > 1 and not is_initial(name_words[-1]):
        last = name_words[-1]
    return first, last, place


# try it on a few made-up endings
for example in ["...our community.Mary Jones [X] Boise, Idaho 83702",
                "Thanks for listening. Sincerely, Dr. Sam Lee Austin, TX 78701",
                "I oppose this rule and encourage the Census Bureau to withdraw it."]:
    print(get_signature(example))

('Mary', 'Jones', 'Boise, Idaho 83702')
('Sam', 'Lee', 'Austin, TX 78701')
(None, None, None)


<div style="background-color:#fff3cd; padding:12px; border-radius:4px">

This cell runs `get_signature()` on every comment and adds the three results to `reg` as new columns: `com_first`, `com_last` and `com_place`. `apply()` runs the function once for each comment, and `apply(pd.Series)` turns each (first, last, place) result into three columns. A comment with no signature gets a blank (`NaN`) in all three. The cell then prints how many comments have a full name (first and last), a first name only, or no name, and how many have a place. It ends by showing the new columns for 10 random comments. The place is whatever the writer gave: sometimes a city and state, sometimes only a state.

</div>

In [24]:
# run get_signature() on every comment; each result is (first, last, place)
signatures = reg["Comment"].apply(get_signature)

# split the three results into three new columns
reg[["com_first", "com_last", "com_place"]] = signatures.apply(pd.Series)

has_first = reg["com_first"].notna()
has_last = reg["com_last"].notna()
print("full name (first and last):", (has_first & has_last).sum())
print("first name only:           ", (has_first & ~has_last).sum())
print("no name found:             ", (~has_first).sum())
print("place found:               ", reg["com_place"].notna().sum())
print("total:                     ", len(reg))

# 10 random comments: the end of the text next to what was found
look = reg.sample(10, random_state=1)[["com_first", "com_last", "com_place"]].copy()
look["end of comment"] = reg.loc[look.index, "Comment"].apply(clean_text).str[-60:]
look

full name (first and last): 8737
first name only:            2616
no name found:              3647
place found:                8748
total:                      15000


,com_first,com_last,com_place,end of comment
7576,Melanie,Howie,"Byron, Georgia 31008","counts our community.Melanie Howie [X] Byron, Georgia 31008"
10509,Jane,NaN,NaN,sus that fairly and accurately counts our community.Jane [X]
4253,NaN,NaN,NaN,"rker and struggling legal citizen, this makes me very angry!"
5150,Dianne,Bruining,Illinois 60448,"and inclusive 2030 Census.Dianne Bruining[x], Illinois 60448"
506,Faye,NaN,Bakersfield CA 93312,"ou,Faye [X]Bakersfield CA 93312Bakersfield, California 93312"
1622,Dolores,Ham,"NOXVILLE, Tennessee 37923","roposal. Thank you,Dolores Ham [x] NOXVILLE, Tennessee 37923"
13684,Lauro,NaN,NaN,"e Census Bureau to withdraw this proposal.Thank you,Lauro[x]"
2145,Sherrill,Futrell,"Davis, California 95618","dering my comment.Sherrill Futrell[x]Davis, California 95618"
9296,Patricia,Foschi,"Santa Fe, NM","m all Americans. Sincerely, Dr. Patricia Foschi Santa Fe, NM"
6900,Donna,Munro,"Bremerton, WA","ted through the Census. Sincerely, Donna Munro Bremerton, WA"


<div style="background-color:#fff3cd; padding:12px; border-radius:4px">

**Check.** A function like this can be wrong without any error message, so we compare it with a person reading the same comments. I read 80 random comments (`reg.sample(80, random_state=21)`) and sorted each into "full name", "first name only" or "no name", then typed the row numbers of the last two groups into the lists below. These 80 were not used to write the rules, so they are a fairer test than the 100 we read earlier. One caution: I read each comment with the program's answer next to it, so my reading may lean towards agreeing with it. The cell sorts the same 80 the same way using the new columns, counts how often the two agree (`pd.crosstab()` is a table of hand against program), and lists every comment where they differ.

</div>

In [25]:
# the row numbers of 80 random comments that I read by hand, and what I decided about each
checked = reg.sample(80, random_state=21)
first_name_only_rows = [1215, 1994, 2052, 4579, 7930, 8253, 9242, 12272, 13048, 14175]
no_name_rows = [269, 385, 3031, 3290, 4687, 5133, 5285, 5372, 5969, 6160, 6421, 6754,
                7721, 7924, 8218, 8498, 9458, 11881, 11957, 12623, 13814, 14838]

# my reading of each comment
by_hand = pd.Series("full name", index=checked.index)
by_hand[first_name_only_rows] = "first name only"
by_hand[no_name_rows] = "no name"

# what the new columns say about the same comments
by_program = pd.Series("full name", index=checked.index)
by_program[checked["com_last"].isna()] = "first name only"
by_program[checked["com_first"].isna()] = "no name"

print("agree:", (by_hand == by_program).sum(), "of", len(checked))
print(pd.crosstab(by_hand, by_program, rownames=["by hand"], colnames=["by program"]))

# every comment where the two disagree
disagree = checked.index[by_hand != by_program]
look = reg.loc[disagree, ["com_first", "com_last", "com_place"]].copy()
look["by hand"] = by_hand[disagree]
look["end of comment"] = reg.loc[disagree, "Comment"].apply(clean_text).str[-70:]
look

agree: 78 of 80
by program       first name only  full name  no name
by hand                                             
first name only               10          0        0
full name                      0         46        2
no name                        0          0       22


,com_first,com_last,com_place,by hand,end of comment
982,NaN,NaN,"Honolulu, Hawaii 96814",full name,"w this proposal.Thank you,B. A. McClintock [X] Honolulu , Hawaii 96814"
3754,NaN,NaN,NaN,full name,See attached file(s)jose lugo
